# DDPM Province + Chiang Rai Tambon Min–Max Scores (Gold) | B.E. 2560–2567

Purpose: presentation-ready maps using **min–max normalization**:  
`score = (x - min) / (max - min)`

Scope (confirmed):
- Province-level maps: `min/max` computed **across all provinces nationally** for each metric
- Chiang Rai tambon maps: `min/max` computed **across all tambons nationally** for each metric
- Metrics: `affected_households_sum`, `affected_people_sum`, `deaths_sum`

Notes / risks:
- Min–max is sensitive to outliers. If the max is extreme, most scores compress near 0.
- This notebook keeps the original raw columns and creates `score_*` columns for mapping.


In [1]:
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

# --- Thai text rendering (best-effort) ---
thai_font_candidates = [
    'TH Sarabun New',
    'Sarabun',
    'Noto Sans Thai',
    'Tahoma',
    'Angsana New',
    'Cordia New',
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
thai_font = next((_f for _f in thai_font_candidates if _f in available_fonts), None)
if thai_font is not None:
    mpl.rcParams['font.family'] = thai_font
mpl.rcParams['axes.unicode_minus'] = False

mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

# Notebook-relative root (data_system/)
BASE_PATH = Path.cwd().resolve().parent.parent  # .../data_system
print('BASE_PATH =', BASE_PATH)

FACT_PATH = BASE_PATH / 'data/2_gold/ddpm/fact_ddpm_tambon_impact_climate_2560_2567.csv'
TAMBON_SHP_PATH = BASE_PATH / 'data/1_silver/dopa/tambon_boundaries_enriched.shp'
PROVINCE_BRONZE_SHP_PATH = BASE_PATH / 'data/0_bronze/dopa/thailanda-administrative-boundary/THA_Province.shp'

METRICS = ['affected_households_sum', 'affected_people_sum', 'deaths_sum']


BASE_PATH = C:\Users\sitth\OracleWorkspace\Arun_Creagy\ψ\incubate\DCCE\CRI\data_system


In [2]:
# --- Load Gold facts + geometries ---
def _read_csv_flex(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='cp874')

def _extract_digits(value: object) -> str:
    if value is None:
        return ''
    s = str(value).strip()
    m = re.search(r'(\d+)', s)
    return m.group(1) if m else ''

def _clean_code_6(series: pd.Series) -> pd.Series:
    s = series.astype(str)
    s = s.str.replace(r'\.0$', '', regex=True).str.strip()
    s = s.map(_extract_digits)
    s = s.str[-6:].fillna('')
    s = s.where(s.str.fullmatch(r'\d{6}', na=False), '')
    s = s.where(~s.isin(['', '000000', 'nan', 'None']), '')
    return s

fact = _read_csv_flex(FACT_PATH)
gdf_tambon = gpd.read_file(TAMBON_SHP_PATH)
gdf_prov = gpd.read_file(PROVINCE_BRONZE_SHP_PATH)

print('fact.shape =', fact.shape)
print('gdf_tambon.shape =', gdf_tambon.shape)
print('gdf_prov.shape =', gdf_prov.shape)

# CRS alignment
gdf_prov = gdf_prov.to_crs(gdf_tambon.crs)

# Clean codes (same approach as the national notebook)
if 'subdistrict_code' not in fact.columns:
    raise KeyError('Missing subdistrict_code in fact')
fact = fact.copy()
fact['subdistrict_code'] = _clean_code_6(fact['subdistrict_code'])
fact = fact.loc[fact['subdistrict_code'].ne('')].copy()
# Province code derived from the first 2 digits of subdistrict_code
fact['province_code'] = fact['subdistrict_code'].str[:2]

if 'subdist_cd' not in gdf_tambon.columns:
    raise KeyError('Expected tambon geometry to contain column subdist_cd')
gdf_tambon = gdf_tambon.copy()
gdf_tambon['subdist_cd'] = _clean_code_6(gdf_tambon['subdist_cd'])
gdf_tambon = gdf_tambon.loc[gdf_tambon['subdist_cd'].ne('')].copy()

# Tambon join (code-only)
map_df = gdf_tambon.merge(fact, left_on='subdist_cd', right_on='subdistrict_code', how='left')
for m in METRICS:
    map_df[m] = pd.to_numeric(map_df[m], errors='coerce').fillna(0.0)
# Ensure derived province_code exists even when join misses
if 'province_code' not in map_df.columns:
    map_df['province_code'] = map_df['subdist_cd'].astype(str).str[:2]
else:
    map_df['province_code'] = map_df['province_code'].fillna(map_df['subdist_cd'].astype(str).str[:2]).astype(str).str.zfill(2)

display(map_df[['subdistrict_code','province_code'] + METRICS].head())


fact.shape = (6967, 13)
gdf_tambon.shape = (8125, 10)
gdf_prov.shape = (77, 4)


,subdistrict_code,province_code,affected_households_sum,affected_people_sum,deaths_sum
0,301012,30,12.0,0.0,0.0
1,430508,43,0.0,0.0,0.0
2,720103,72,20.0,0.0,0.0
3,360702,36,84.0,0.0,0.0
4,650610,65,325.0,33.0,0.0


In [3]:
def minmax_score(series: pd.Series, *, clip_min: float | None = None, clip_max: float | None = None) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce').fillna(0.0).astype(float)
    if clip_min is not None or clip_max is not None:
        s = s.clip(lower=clip_min, upper=clip_max)
    mn = float(s.min()) if len(s) else 0.0
    mx = float(s.max()) if len(s) else 0.0
    denom = (mx - mn)
    if denom <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    return (s - mn) / denom

# Create tambon-level score columns using national min/max across all tambons
for m in METRICS:
    map_df[f'score_{m}'] = minmax_score(map_df[m])

map_df[[f'score_{m}' for m in METRICS]].describe()


,score_affected_households_sum,score_affected_people_sum,score_deaths_sum
count,8125.000000,8125.000000,8125.000000
mean,0.014523,0.006179,0.004767
std,0.040763,0.028452,0.029077
min,0.000000,0.000000,0.000000
25%,0.000940,0.000000,0.000000
50%,0.004100,0.000000,0.000000
75%,0.011788,0.001211,0.000000
max,1.000000,1.000000,1.000000


## A) Province-level maps (min–max score)
Aggregate tambon facts to province totals, then min–max normalize across provinces nationally (per metric), and join to province boundaries for choropleths.

In [4]:
# Province aggregation from Gold facts
prov_stats = (
    fact.groupby('province_code', as_index=False)[METRICS]
        .sum(numeric_only=True)
)

# Province-level score columns using national min/max across all provinces
for m in METRICS:
    prov_stats[f'score_{m}'] = minmax_score(prov_stats[m])

display(prov_stats.sort_values('score_affected_households_sum', ascending=False).head(10))


,province_code,affected_households_sum,affected_people_sum,deaths_sum,score_affected_households_sum,score_affected_people_sum,score_deaths_sum
38,50,506865.0,299860.0,19.0,1.000000,1.000000,0.791667
63,80,214024.0,28978.0,21.0,0.421863,0.096638,0.875000
76,96,149279.0,231749.0,7.0,0.294041,0.772857,0.291667
45,57,130760.0,40012.0,10.0,0.257480,0.133436,0.416667
34,46,85089.0,133300.0,4.0,0.167315,0.444541,0.166667
67,84,73264.0,37999.0,18.0,0.143969,0.126722,0.750000
51,64,61566.0,68624.0,4.0,0.120875,0.228853,0.166667
74,94,52748.0,42575.0,24.0,0.103466,0.141983,1.000000
40,52,50475.0,9218.0,7.0,0.098978,0.030741,0.291667
18,30,47464.0,23233.0,4.0,0.093034,0.077479,0.166667


In [5]:
# Standardize province code column in province geometry
prov_code_col = None
for c in ['P_CODE', 'p_code', 'prov_code', 'province_code', 'PROV_CODE']:
    if c in gdf_prov.columns:
        prov_code_col = c
        break
if prov_code_col is None:
    raise KeyError('Cannot find a province code column in province geometry. Available columns: ' + str(list(gdf_prov.columns)))

gdf_prov = gdf_prov.copy()
gdf_prov['province_code'] = gdf_prov[prov_code_col].astype(str).str.extract(r'(\d+)')[0].str.zfill(2)

prov_map = gdf_prov.merge(prov_stats, on='province_code', how='left')
for m in METRICS:
    prov_map[f'score_{m}'] = pd.to_numeric(prov_map[f'score_{m}'], errors='coerce').fillna(0.0)

print('prov_map.shape =', prov_map.shape)


KeyError: "Cannot find a province code column in province geometry. Available columns: ['P_NAME_T', 'P_NAME_E', 'Area_km2', 'geometry']"

In [12]:
def plot_province_score(gdf: gpd.GeoDataFrame, score_col: str, title: str, *, cmap: str = 'Reds') -> None:
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.set_title(title)
    ax.set_axis_off()

    gdf.plot(
        column=score_col,
        ax=ax,
        cmap=cmap,
        legend=True,
        legend_kwds={'shrink': 0.6, 'fraction': 0.04, 'pad': 0.02},
        linewidth=0.4,
        edgecolor='#222222',
        vmin=0,
        vmax=1,
        missing_kwds={'color': '#f0f0f0', 'edgecolor': 'none', 'label': 'No data'},
    )
    plt.show()

plot_province_score(prov_map, 'score_affected_households_sum', 'Province score (min–max): affected households | 2560–2567')
plot_province_score(prov_map, 'score_affected_people_sum', 'Province score (min–max): affected people | 2560–2567')
plot_province_score(prov_map, 'score_deaths_sum', 'Province score (min–max): deaths | 2560–2567')


NameError: name 'prov_map' is not defined

## B) Chiang Rai tambon maps (min–max score)
Filter tambons by subdistrict code prefix `57` (Chiang Rai) and map the min–max scores computed across **all tambons nationally**.

In [ ]:
cr = map_df[map_df['subdistrict_code'].astype(str).str.startswith('57')].copy()
print('Chiang Rai tambons:', cr.shape[0])

def plot_tambon_score(gdf: gpd.GeoDataFrame, score_col: str, title: str, *, cmap: str = 'Reds', prov_borders: gpd.GeoDataFrame | None = None) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    ax.set_title(title)
    ax.set_axis_off()

    gdf.plot(
        column=score_col,
        ax=ax,
        cmap=cmap,
        legend=True,
        legend_kwds={'shrink': 0.6, 'fraction': 0.04, 'pad': 0.02},
        linewidth=0.2,
        edgecolor='#444444',
        vmin=0,
        vmax=1,
        missing_kwds={'color': '#f0f0f0', 'edgecolor': 'none', 'label': 'No data'},
    )

    if prov_borders is not None:
        try:
            prov_borders.boundary.plot(ax=ax, color='#111111', linewidth=0.6, alpha=0.5)
        except Exception:
            pass

    plt.show()

plot_tambon_score(cr, 'score_affected_households_sum', 'Chiang Rai tambon score (min–max national): affected households | 2560–2567')
plot_tambon_score(cr, 'score_affected_people_sum', 'Chiang Rai tambon score (min–max national): affected people | 2560–2567')
plot_tambon_score(cr, 'score_deaths_sum', 'Chiang Rai tambon score (min–max national): deaths | 2560–2567')
